# 机器学习统计学（Statistics for Machine Learning）

对应课程：`phases/01-math-foundations/15-statistics-for-ml`

> 统计学能告诉你：模型是真的有效，还是只是运气好。

本 notebook 把 `statistics.py` 里的核心函数拆开：每个函数一组中文注释，后面跟一小段可运行实验。完整打印型 demo 仍在 `statistics.py`。

**贯穿全课的模式：** 描述数据 → 相关 ≠ 因果 → 假设检验给 p 值 → 效应量和多重检验告诉你 p 小不等于有用。


## 学习目标（Learning Objectives）

- 从零算描述统计、Pearson/Spearman、协方差矩阵
- 做 t 检验和卡方，正确读 p 值和置信区间
- 用 bootstrap 给任意指标做置信区间，不靠分布假设
- 用效应量区分「统计显著」和「实际有用」


## 0. 依赖


In [1]:
import math
import random

random.seed(42)


## 1. 描述统计：中心、散度、分位

样本方差除以 $n-1$（无偏）；总体方差除以 $n$。IQR = Q3 − Q1，对离群点比标准差稳。


In [2]:
def mean(data):
    return sum(data) / len(data)


def median(data):
    s = sorted(data)
    n = len(s)
    mid = n // 2
    return (s[mid - 1] + s[mid]) / 2 if n % 2 == 0 else s[mid]


def variance(data, sample=True):
    n = len(data)
    m = mean(data)
    total = sum((x - m) ** 2 for x in data)
    return total / (n - 1) if sample and n > 1 else total / n


def std_dev(data, sample=True):
    return math.sqrt(variance(data, sample))


def percentile(data, p):
    """线性插值分位数。"""
    s = sorted(data)
    n = len(s)
    k = (p / 100) * (n - 1)
    f, c = math.floor(k), math.ceil(k)
    if f == c:
        return s[int(k)]
    return s[f] * (c - k) + s[c] * (k - f)


def iqr(data):
    return percentile(data, 75) - percentile(data, 25)


xs = [1, 2, 2, 3, 100]
print("mean", mean(xs), "median", median(xs), "  (均值被 100 拉走)")
print("sample var", round(variance(xs), 2), "IQR", iqr(xs))


mean 21.6 median 2   (均值被 100 拉走)
sample var 1921.3 IQR 1


## 2. Pearson vs Spearman

Pearson 量线性相关。Spearman 先把数据换成秩再算 Pearson，对单调非线性更稳。


In [3]:
def covariance(x, y, sample=True):
    n = len(x)
    mx, my = mean(x), mean(y)
    total = sum((xi - mx) * (yi - my) for xi, yi in zip(x, y))
    return total / (n - 1) if sample and n > 1 else total / n


def pearson_correlation(x, y):
    n = len(x)
    mx, my = mean(x), mean(y)
    sx, sy = std_dev(x, sample=False), std_dev(y, sample=False)
    if sx == 0 or sy == 0:
        return 0.0
    cov = sum((xi - mx) * (yi - my) for xi, yi in zip(x, y)) / n
    return cov / (sx * sy)


def rank_data(data):
    """平均秩处理并列。"""
    indexed = sorted(enumerate(data), key=lambda pair: pair[1])
    ranks = [0.0] * len(data)
    i = 0
    while i < len(indexed):
        j = i
        while j < len(indexed) - 1 and indexed[j + 1][1] == indexed[i][1]:
            j += 1
        avg_rank = (i + j) / 2.0 + 1.0
        for k in range(i, j + 1):
            ranks[indexed[k][0]] = avg_rank
        i = j + 1
    return ranks


def spearman_correlation(x, y):
    return pearson_correlation(rank_data(x), rank_data(y))


x = [1, 2, 3, 4, 5, 6]
y_lin = [2, 4, 6, 8, 10, 12]
y_mono = [1, 4, 9, 16, 25, 36]  # 平方，单调但非线性
print("线性  Pearson", pearson_correlation(x, y_lin), "Spearman", spearman_correlation(x, y_lin))
print("平方  Pearson", round(pearson_correlation(x, y_mono), 4),
      "Spearman", round(spearman_correlation(x, y_mono), 4))


线性  Pearson 1.0 Spearman 1.0
平方  Pearson 0.9789 Spearman 1.0


## 3. t 检验：均值是不是碰巧不同

单样本：$t=(\bar x-\mu_0)/(s/\sqrt n)$。双样本用 Welch（不假设等方差）。p 值来自 t 分布 CDF（正则化不完全 Beta）。


In [4]:
def t_statistic_one_sample(data, mu_0):
    n = len(data)
    return (mean(data) - mu_0) / (std_dev(data, sample=True) / math.sqrt(n))


def t_statistic_two_sample(data1, data2):
    n1, n2 = len(data1), len(data2)
    se = math.sqrt(variance(data1) / n1 + variance(data2) / n2)
    return 0.0 if se == 0 else (mean(data1) - mean(data2)) / se


def welch_df(data1, data2):
    n1, n2 = len(data1), len(data2)
    v1, v2 = variance(data1), variance(data2)
    num = (v1 / n1 + v2 / n2) ** 2
    denom = (v1 / n1) ** 2 / (n1 - 1) + (v2 / n2) ** 2 / (n2 - 1)
    return (n1 + n2 - 2) if denom == 0 else num / denom


def _log_beta_function(a, b):
    return math.lgamma(a) + math.lgamma(b) - math.lgamma(a + b)


def _beta_continued_fraction(x, a, b, max_iter=300, tol=1e-15):
    tiny = 1e-300
    qab, qap, qam = a + b, a + 1.0, a - 1.0
    c, d = 1.0, 1.0 - qab * x / qap
    d = 1.0 / (tiny if abs(d) < tiny else d)
    h = d
    for m in range(1, max_iter + 1):
        m2 = 2 * m
        for num in (m * (b - m) * x / ((qam + m2) * (a + m2)),
                    -(a + m) * (qab + m) * x / ((a + m2) * (qap + m2))):
            d = 1.0 + num * d
            d = 1.0 / (tiny if abs(d) < tiny else d)
            c = 1.0 + num / c
            c = tiny if abs(c) < tiny else c
            h *= d * c
        if abs(d * c - 1.0) < tol:
            break
    return h


def _regularized_beta(x, a, b):
    if x <= 0:
        return 0.0
    if x >= 1:
        return 1.0
    front = math.exp(a * math.log(x) + b * math.log(1.0 - x) - _log_beta_function(a, b))
    if x < (a + 1.0) / (a + b + 2.0):
        return front * _beta_continued_fraction(x, a, b) / a
    return 1.0 - front * _beta_continued_fraction(1.0 - x, b, a) / b


def t_cdf_approx(t_val, df):
    x = df / (df + t_val * t_val)
    if t_val < 0:
        return 0.5 * _regularized_beta(x, df / 2, 0.5)
    return 1.0 - 0.5 * _regularized_beta(x, df / 2, 0.5)


def p_value_two_sided(t_val, df):
    return 2.0 * (1.0 - t_cdf_approx(abs(t_val), df))


def one_sample_ttest(data, mu_0=0):
    t = t_statistic_one_sample(data, mu_0)
    df = len(data) - 1
    return {"t_statistic": t, "df": df, "p_value": p_value_two_sided(t, df)}


def two_sample_ttest(data1, data2):
    t = t_statistic_two_sample(data1, data2)
    df = welch_df(data1, data2)
    return {"t_statistic": t, "df": df, "p_value": p_value_two_sided(t, df)}


a = [5.1, 5.0, 5.2, 4.9, 5.3]
b = [6.0, 6.2, 5.8, 6.1, 5.9]
print("单样本 vs 5:", {k: round(v, 4) if isinstance(v, float) else v for k, v in one_sample_ttest(a, 5.0).items()})
print("两样本 A vs B:", {k: round(v, 4) if isinstance(v, float) else v for k, v in two_sample_ttest(a, b).items()})


单样本 vs 5: {'t_statistic': 1.4142, 'df': 4, 'p_value': 0.2302}
两样本 A vs B: {'t_statistic': -9.0, 'df': 8.0, 'p_value': 0.0}


## 4. 卡方：观察频数 vs 期望频数

$$
\chi^2=\sum (O_i-E_i)^2/E_i
$$


In [5]:
def _gamma_series(a, x, max_iter=1000, tol=1e-16):
    term = total = 1.0 / a
    ap = a
    for _ in range(max_iter):
        ap += 1.0
        term *= x / ap
        total += term
        if abs(term) < abs(total) * tol:
            break
    return total * math.exp(-x + a * math.log(x) - math.lgamma(a))


def _gamma_continued_fraction(a, x, max_iter=1000, tol=1e-16):
    tiny = 1e-300
    b = x + 1.0 - a
    c, d = 1.0 / tiny, 1.0 / b
    h = d
    for i in range(1, max_iter + 1):
        an = -i * (i - a)
        b += 2.0
        d = an * d + b
        d = tiny if abs(d) < tiny else d
        d = 1.0 / d
        c = b + an / c
        c = tiny if abs(c) < tiny else c
        delta = d * c
        h *= delta
        if abs(delta - 1.0) < tol:
            break
    return h * math.exp(-x + a * math.log(x) - math.lgamma(a))


def chi_squared_p_value(chi2, df):
    if chi2 <= 0:
        return 1.0
    a, x = df / 2.0, chi2 / 2.0
    cdf = _gamma_series(a, x) if x < a + 1.0 else 1.0 - _gamma_continued_fraction(a, x)
    return 1.0 - cdf


def chi_squared_test(observed, expected):
    chi2 = sum((o - e) ** 2 / e for o, e in zip(observed, expected) if e > 0)
    df = len(observed) - 1
    return {"chi2": chi2, "df": df, "p_value": chi_squared_p_value(chi2, df)}


# 公平骰子？观察 [10,8,12,9,11,10]，期望全是 10
print(chi_squared_test([10, 8, 12, 9, 11, 10], [10] * 6))


{'chi2': 1.0, 'df': 5, 'p_value': 0.9625657732472964}


## 5. Bootstrap：用重抽样估置信区间

有放回抽很多次，看统计量的分布。不必假设正态。notebook 里用 400 次（源码默认 5000）。


In [6]:
def bootstrap_statistic(data, stat_func, n_bootstrap=400, ci=95):
    n = len(data)
    stats = []
    for _ in range(n_bootstrap):
        sample = [data[random.randint(0, n - 1)] for _ in range(n)]
        stats.append(stat_func(sample))
    stats.sort()
    lo = (100 - ci) / 2
    return {
        "estimate": stat_func(data),
        "ci_lower": percentile(stats, lo),
        "ci_upper": percentile(stats, 100 - lo),
        "std_error": std_dev(stats, sample=True),
    }


data = [2.1, 2.4, 1.9, 2.2, 2.0, 2.3, 2.5, 1.8]
res = bootstrap_statistic(data, mean)
print({k: round(v, 4) for k, v in res.items()})


{'estimate': 2.15, 'ci_lower': 1.975, 'ci_upper': 2.3, 'std_error': 0.0818}


## 6. 效应量与 Bonferroni

p 小可能只是样本大。Cohen's d 看差了几倍标准差。Bonferroni：做 $m$ 次检验时把阈值改成 $\alpha/m$，否则假阳性会堆起来。


In [7]:
def cohens_d(data1, data2):
    n1, n2 = len(data1), len(data2)
    pooled = math.sqrt(((n1 - 1) * variance(data1) + (n2 - 1) * variance(data2)) / (n1 + n2 - 2))
    return 0.0 if pooled == 0 else (mean(data1) - mean(data2)) / pooled


def interpret_cohens_d(d):
    d = abs(d)
    if d < 0.2:
        return "negligible"
    if d < 0.5:
        return "small"
    if d < 0.8:
        return "medium"
    return "large"


def bonferroni_correction(p_values, alpha=0.05):
    adj = alpha / len(p_values)
    return [{"original_p": p, "adjusted_alpha": adj, "significant": p < adj} for p in p_values]


d = cohens_d(a, b)
print("Cohen's d", round(d, 3), interpret_cohens_d(d))
ps = [0.01, 0.04, 0.03, 0.20]
print("原始 alpha=0.05 会显著", sum(p < 0.05 for p in ps), "个")
print("Bonferroni 后显著", sum(r["significant"] for r in bonferroni_correction(ps)), "个")


Cohen's d -5.692 large
原始 alpha=0.05 会显著 3 个
Bonferroni 后显著 1 个


## 7. 协方差矩阵，以及显著 ≠ 有用（学习目标）

协方差矩阵 $(i,j)$ 是第 i、j 维一起怎么变。大样本时，真效应只有 0.1 也能把 p 压到 0.05 以下，Cohen's d 仍然 negligible。


In [8]:
def covariance_matrix(cols):
    """cols[d][n]：每个特征一条序列。"""
    d, n = len(cols), len(cols[0])
    means = [mean(cols[i]) for i in range(d)]
    mat = [[0.0] * d for _ in range(d)]
    for i in range(d):
        for j in range(i, d):
            cov = sum((cols[i][k] - means[i]) * (cols[j][k] - means[j]) for k in range(n)) / (n - 1)
            mat[i][j] = mat[j][i] = cov
    return mat


x = [1.0, 2.0, 3.0, 4.0, 5.0]
y = [2.0, 4.0, 6.0, 8.0, 11.0]
print("协方差矩阵:\n", [[round(v, 3) for v in row] for row in covariance_matrix([x, y])])


def generate_normal(n, mu=0, sigma=1):
    out = []
    while len(out) < n:
        u1 = random.random() or 1e-12
        u2 = random.random()
        z0 = math.sqrt(-2 * math.log(u1)) * math.cos(2 * math.pi * u2)
        out.append(mu + sigma * z0)
    return out[:n]


def report(n, effect=0.1):
    a = generate_normal(n, 50, 10)
    b = generate_normal(n, 50 + effect, 10)
    p = two_sample_ttest(a, b)["p_value"]
    d = cohens_d(a, b)
    print(f"n={n:<6}  p={p:.4g}  显著={p<0.05}  d={d:.3f} ({interpret_cohens_d(d)})")


print("真效应 mean 只差 0.1、sigma=10:")
report(30)
report(8000)


协方差矩阵:
 [[2.5, 5.5], [5.5, 12.2]]
真效应 mean 只差 0.1、sigma=10:
n=30      p=0.5063  显著=False  d=-0.173 (negligible)
n=8000    p=0.1952  显著=False  d=-0.020 (negligible)


## 对照表

| 函数 | 角色 |
|------|------|
| mean / median / var / IQR | 描述数据；中位数抗离群点 |
| Pearson / Spearman | 线性 vs 单调相关 |
| t 检验 | 均值差是不是噪声 |
| 卡方 | 频数是否符合期望 |
| bootstrap | 不靠正态假设的置信区间 |
| Cohen's d | 显著 ≠ 有用 |
| Bonferroni | 多次检验会灌水假阳性 |

```bash
python statistics.py
```
